In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Xem các sheet trong file
file_path = "final.xlsx"
excel_file = pd.ExcelFile(file_path)

print("Sheets:", excel_file.sheet_names)

# Đọc sheet đầu tiên
df_raw = pd.read_excel(file_path, sheet_name=excel_file.sheet_names[0])

print("Kích thước dữ liệu:", df_raw.shape)
display(df_raw.head())

In [15]:
summary = pd.DataFrame({
    "Data Type": df_raw.dtypes.astype(str),
    "Non-Null": df_raw.notna().sum(),
    "Null": df_raw.isna().sum(),
    "Null (%)": (df_raw.isna().mean() * 100).round(2),
    "Unique Values": df_raw.nunique()
})

display(summary)

,Data Type,Non-Null,Null,Null (%),Unique Values
MaHoSo_TS,str,22874,0,0.00,22870
DiemTongCong,float64,22265,609,2.66,216
NamTotNghiep,float64,22148,726,3.17,59
IDSinhVien,int64,22874,0,0.00,22874
MaSinhVien,int64,22874,0,0.00,22874
Ten,str,22874,0,0.00,801
GioiTinh,bool,22874,0,0.00,2
NgaySinh2,str,22874,0,0.00,4538
KhoaHoc,int64,22874,0,0.00,7
IDNganh,int64,22874,0,0.00,81


In [16]:
display(df_raw.describe().T)

,count,mean,min,25%,50%,75%,max,std
DiemTongCong,22265.0,6.996441,0.0,6.3,7.0,7.5,100.0,2.920783
NamTotNghiep,22148.0,2018.916832,-16.0,2019.0,2021.0,2022.0,2202.0,57.348206
IDSinhVien,22874.0,41762.467255,13862.0,30223.5,39073.5,51571.75,71051.0,13388.19059
MaSinhVien,22874.0,21161916.331687,17001843.0,20001881.5,21001991.0,22004657.75,25002962.0,1664255.59779
KhoaHoc,22874.0,2021.162805,2019.0,2020.0,2021.0,2022.0,2025.0,1.663406
IDNganh,22874.0,112.487672,71.0,90.0,102.0,135.0,197.0,32.471354
IDLopHoc,22874.0,1664.232797,938.0,1297.0,1583.0,2024.75,2716.0,453.089342
NgayNhapHoc,22311,2021-10-14 04:54:48.438319,2003-07-11 00:00:00,2020-08-03 00:00:00,2021-08-26 00:00:00,2022-10-03 00:00:00,2025-10-31 00:00:00,NaN
DiemTBHocLuc_HK1,22874.0,5.164241,0.0,4.02,5.99,6.92,10.0,2.483764
DiemTBTinChi_HK1,22874.0,1.734446,0.0,0.92,2.0,2.55,4.0,1.072698


In [17]:
# Phân bố target
if "NhomTrangThai" in df_raw.columns:
    display(df_raw["NhomTrangThai"].value_counts(dropna=False))

# Kiểm tra ID sinh viên trùng
if "IDSinhVien" in df_raw.columns:
    print("Số IDSinhVien trùng:",
          df_raw["IDSinhVien"].duplicated().sum())

NhomTrangThai
1    12207
2    10667
Name: count, dtype: int64

Số IDSinhVien trùng: 0


In [18]:
df = df_raw.copy()

# Chỉ giữ sinh viên không bị đánh dấu nghỉ học quá sớm
if "NghiHocHK1" in df.columns:
    df["NghiHocHK1"] = pd.to_numeric(
        df["NghiHocHK1"], errors="coerce"
    )
    df = df[df["NghiHocHK1"] == 0].copy()

print("Sau khi lọc NghiHocHK1:", df.shape)

Sau khi lọc NghiHocHK1: (21478, 38)


In [19]:
# 1. Xem unique values trước khi chuyển đổi
for col in ["GhiChu_HK1", "GhiChu_HK2"]:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).to_frame("SoLuong"))

# 2. Đổi tên cột
df.rename(columns={
    "GhiChu_HK1": "CanhCaoHocVu_HK1",
    "GhiChu_HK2": "CanhCaoHocVu_HK2"
}, inplace=True)

# 3. Có chữ "học vụ" -> 1, không có -> 0
for col in ["CanhCaoHocVu_HK1", "CanhCaoHocVu_HK2"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.contains("học vụ", case=False, na=False)
        .astype("int8")
    )

# 4. Kiểm tra kết quả
for col in ["CanhCaoHocVu_HK1", "CanhCaoHocVu_HK2"]:
    print(f"\n--- Phân bố {col} ---")
    display(df[col].value_counts().to_frame("SoLuong"))


--- GhiChu_HK1 ---


,SoLuong
GhiChu_HK1,
Học tiếp,17642
Cảnh cáo học vụ lần 1,3836



--- GhiChu_HK2 ---


,SoLuong
GhiChu_HK2,
Học tiếp,14067
Cảnh cáo học vụ lần 1,3385
NaN,2185
Cảnh cáo học vụ lần 2,1841



--- Phân bố CanhCaoHocVu_HK1 ---


,SoLuong
CanhCaoHocVu_HK1,
0,17642
1,3836



--- Phân bố CanhCaoHocVu_HK2 ---


,SoLuong
CanhCaoHocVu_HK2,
0,16252
1,5226


In [20]:
# Chuyển NgaySinh2 sang datetime
ngay_sinh = pd.to_datetime(
    df["NgaySinh2"],
    errors="coerce",
    dayfirst=True
)

# Tạo cột tuổi tại năm khóa học
df["Tuoi"] = (
    pd.to_numeric(df["KhoaHoc"], errors="coerce")
    - ngay_sinh.dt.year
).astype("Int64")

In [21]:
display(df[["NgaySinh2", "KhoaHoc", "Tuoi"]].head(10))

print("Số giá trị tuổi bị thiếu:", df["Tuoi"].isna().sum())
display(df["Tuoi"].describe())

,NgaySinh2,KhoaHoc,Tuoi
0,06/01/1996,2019,23
1,14/07/2001,2019,18
2,19/12/2000,2020,20
3,04/03/1997,2019,22
4,29/02/2000,2019,19
6,07/05/1999,2019,20
7,26/11/1999,2019,20
8,09/12/2000,2019,19
9,30/10/2004,2019,15
10,01/07/2004,2019,15


Số giá trị tuổi bị thiếu: 1


count      21477.0
mean     17.539507
std       3.077049
min           -5.0
25%           15.0
50%           18.0
75%           18.0
max           60.0
Name: Tuoi, dtype: Float64

In [23]:
# Xóa khoảng trắng thừa ở đầu/cuối
df["TenTinh"] = df["TenTinh"].str.strip()

# Chuẩn hóa các tên khác cách viết nhưng cùng địa phương
ten_tinh_mapping = {
    # TP. Hồ Chí Minh
    "Tp. Hồ Chí Minh": "Hồ Chí Minh",
    "Thành Phố Hồ Chí Minh": "Hồ Chí Minh",

    # Các giá trị có tiền tố "Tỉnh"
    "Tỉnh Ninh Bình": "Ninh Bình",
    "Tỉnh Hưng Yên": "Hưng Yên",
    "Tỉnh Đồng Nai": "Đồng Nai",
    "Tỉnh An Giang": "An Giang",
    "Tỉnh Vĩnh Long": "Vĩnh Long",
    "Tỉnh Quảng Ngãi": "Quảng Ngãi",
    "Tỉnh Lâm Đồng": "Lâm Đồng",

    # Chuẩn hóa chính tả
    "Thanh Hoá": "Thanh Hóa",
    "Khánh Hoà": "Khánh Hòa",
    "Hoà Bình": "Hòa Bình",
    "Đăk Nông": "Đắk Nông",
    "Bắc Cạn": "Bắc Kạn",

    # Chuẩn hóa khoảng trắng
    "Bà Rịa-Vũng Tàu": "Bà Rịa - Vũng Tàu",
    "Thừa Thiên -Huế": "Huế",
    "thành phố Huế": "Huế",

    # Hà Tây cũ hiện thuộc Hà Nội
    "Hà Nội (Hà tây cũ)": "Hà Nội"
}

df["TenTinh"] = df["TenTinh"].replace(ten_tinh_mapping)

In [24]:
print("Số unique trước chuẩn hóa:",
      len(ten_tinh_mapping))

print("Số unique sau chuẩn hóa:",
      df["TenTinh"].nunique(dropna=False))

display(
    df["TenTinh"]
    .value_counts(dropna=False)
    .to_frame("SoLuong")
)

Số unique trước chuẩn hóa: 18
Số unique sau chuẩn hóa: 66


,SoLuong
TenTinh,
Hồ Chí Minh,8172
Long An,1228
Quảng Ngãi,802
Bến Tre,799
Tiền Giang,760
Tây Ninh,606
Bình Định,547
Bình Thuận,520
Quảng Nam,442


In [36]:
display(
    df.loc[
        (df["Tuoi"] < 14) | (df["Tuoi"] > 60),
        ["NgaySinh2", "KhoaHoc", "Tuoi"]
    ].sort_values("Tuoi")
)

display(
    df.loc[
        (df["DiemTongCong"] == 0) | (df["DiemTongCong"] > 10),
        ["DiemTongCong", "KhoaHoc", "TenNganh"]
    ].sort_values("DiemTongCong")
)

,NgaySinh2,KhoaHoc,Tuoi


,DiemTongCong,KhoaHoc,TenNganh


In [34]:
#biến thành nan

df["DiemTongCong"] = pd.to_numeric(
    df["DiemTongCong"], errors="coerce"
)

df.loc[
    (df["DiemTongCong"] == 0) | (df["DiemTongCong"] > 10),
    "DiemTongCong"
] = np.nan

In [38]:
# Chuyển tuổi âm thành NaN
df.loc[df["Tuoi"] < 0, "Tuoi"] = np.nan

# Tính median từ các tuổi hợp lệ
median_tuoi = df.loc[df["Tuoi"] >= 0, "Tuoi"].median()

# Điền median cho các giá trị thiếu
df["Tuoi"] = df["Tuoi"].fillna(median_tuoi)

Median tuổi: 18.0
Số giá trị tuổi còn thiếu: 0


In [42]:
df["TenKhuVuc"] = df["TenKhuVuc"].fillna("Không xác định")
df["TenTinh"] = df["TenTinh"].fillna("Không xác định")

In [43]:
# ĐTB học lực HK1 ngoài khoảng 0–10
display(df.loc[
    ~df["DiemTBHocLuc_HK1"].between(0, 10),
    ["DiemTBHocLuc_HK1"]
])

# Tín chỉ không đạt lớn hơn tín chỉ đăng ký
display(df.loc[
    df["SoTCKhongDat_HK1"] > df["STCDangKyHocKy_HK1"],
    ["STCDangKyHocKy_HK1", "SoTCKhongDat_HK1"]
])

,DiemTBHocLuc_HK1


,STCDangKyHocKy_HK1,SoTCKhongDat_HK1


In [49]:
df.to_excel("Cleaned_FINAL.xlsx", index = False)

In [ ]:
#

#CLEANING DF


#

In [ ]:
drop_cols = [
    # Định danh cá nhân
    "MaHoSo_TS",
    "IDSinhVien",
    "MaSinhVien",
    "Ten",
    "NamTotNghiep",
    "NgayNhapHoc",
    

    # ID đã có cột tên tương ứng
    "IDNganh",
    "IDLopHoc",

    # Ghi chú tự do và trạng thái gốc
    "GhiChu",
    "TenTrangThai",

    # Chỉ dùng để lọc, không dùng làm feature
    "NghiHocHK1",
    "CoHocHK2",
    "CoHocHK3",

    # Thông tin HK2 xảy ra sau mốc dự đoán
    "DiemTBHocLuc_HK2",
    "XepLoaiHocLuc_HK2",
    "DiemTBTinChi_HK2",
    "GhiChu_HK2",
    "STCDangKyHocKy_HK2",
    "SoTCKhongDat_HK2"
]

df_model = df.drop(columns=drop_cols, errors="ignore")

print("Kích thước df_model:", df_model.shape)
display(df_model.head())

In [45]:
summary = pd.DataFrame({
    "Data Type": df_model.dtypes.astype(str),
    "Non-Null": df_model.notna().sum(),
    "Null": df_model.isna().sum(),
    "Null (%)": (df_model.isna().mean() * 100).round(2),
    "Unique Values": df_model.nunique()
})

display(summary)

,Data Type,Non-Null,Null,Null (%),Unique Values
DiemTongCong,float64,20690,788,3.67,176
GioiTinh,bool,21478,0,0.00,2
NgaySinh2,str,21478,0,0.00,4460
KhoaHoc,int64,21478,0,0.00,7
TenNganh,str,21478,0,0.00,47
TenKhuVuc,str,21478,0,0.00,5
TenTinh,str,21478,0,0.00,66
TenHeDaoTao,str,21478,0,0.00,2
TenLoaiHinhDT,str,21478,0,0.00,12
DiemTBHocLuc_HK1,float64,21478,0,0.00,939


In [46]:
display(df_model.describe().T)

,count,mean,std,min,25%,50%,75%,max
DiemTongCong,20690.0,6.922816,0.848469,2.0,6.4,7.0,7.5,10.0
KhoaHoc,21478.0,2021.136419,1.655735,2019.0,2020.0,2021.0,2022.0,2025.0
DiemTBHocLuc_HK1,21478.0,5.446599,2.262762,0.0,4.7,6.1,6.98,10.0
DiemTBTinChi_HK1,21478.0,1.837765,1.018319,0.0,1.2,2.0,2.6,4.0
CanhCaoHocVu_HK1,21478.0,0.178601,0.383027,0.0,0.0,0.0,0.0,1.0
STCDangKyHocKy_HK1,21478.0,15.551914,4.722961,2.0,13.0,14.0,18.0,50.0
SoTCKhongDat_HK1,21478.0,2.826427,4.84955,0.0,0.0,0.0,4.0,50.0
CanhCaoHocVu_HK2,21478.0,0.243319,0.429096,0.0,0.0,0.0,0.0,1.0
DiemHKDot1,18316.0,61.351223,13.952076,0.0,54.0,61.0,70.0,100.0
NhomTrangThai,21478.0,1.431651,0.495318,1.0,1.0,1.0,2.0,2.0
